# All-MoE Induction Eval

Follow-on to the decode/cot/moe **archetype** study in `notebooks/periodic/induction_eval.ipynb`. That experiment controls for reasoning *archetype* (dense-non-reasoning / dense-reasoning / non-reasoning-MoE). Having established those effects, this notebook holds architecture class **constant -- all three models are Mixture-of-Experts** -- and compares three modern open-weight MoE generalists head-to-head on the *identical* periodic induction quizzes (same template, `PeriodicConfig(n=9, labels=9)`, `BASE_SEED=1776`), so results line up directly against the archetype study.

**Trio (all MoE), highest-precision OFFICIAL release each:**
- `qwen3.5-397b-a17b` -> `Qwen/Qwen3.5-397B-A17B` (**BF16**, 397B/17B)
- `nemotron-3-super-120b-a12b` -> `nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16` (**BF16**, 120B/12B)
- `gpt-oss-120b` -> `openai/gpt-oss-120b` (**MXFP4** -- its only/native official precision, 120B/5.1B)

the strongest open-weight generalists with a measured Lean 4 miniF2F number in the UW study (arXiv 2606.05632). All three repos are ungated (verified via the HF models API).

**⚠️ Before a LIVE run:**
1. **Instance = p5e, not p5.** BF16 Qwen3.5-397B is ~794 GB, which does NOT fit p5 (640 GB); `keys.env` pins `EC2_INSTANCE_TYPES=p5e.48xlarge` (1128 GB). Models serve one at a time (the harness swaps vLLM between them), so the box only needs to hold the largest single model. To keep a p5 fallback, switch the `qwen3.5-397b-a17b` spec to FP8 (~397 GB) in `smolbench/evals/ec2.py` and drop the pin.
2. **vLLM image.** These are 2026 architectures (Qwen3.5 = hybrid Gated-DeltaNet + MoE, multimodal; Nemotron-3; gpt-oss harmony). Set `EC2_VLLM_IMAGE` to a current release that supports all three (see the `keys.env` TODO) -- the default image predates them.
3. **Precision is heterogeneous by design** -- BF16 / BF16 / MXFP4 (GPT-OSS ships MXFP4-only, so it can't match the others). Each is the highest-precision official build; carry the mixed precision as a cross-model caveat.
4. **No reasoning contrast.** All three can reason, but this study runs each in a single default mode (8192-token completion headroom) and just checks whether the graded integer answer is correct -- it is *not* a reasoning on/off comparison.
5. **Power.** `periodic_moe/power_analysis.py` adapts the sibling's harmonic-stratified CMH sizing to this 3-MoE contrast; like the sibling it sizes R *from a pilot*, so run the pilot (seed 1776) first.

All lifecycle/harness plumbing lives in `smolbench.induction.experiment.InductionExperiment`; see the sibling notebook's intro for the full replicate / resume / cost mechanics. **Cost:** provisioning spins up one large EC2 spot instance (~$30-45/h for the p5e family) with an idle watchdog + max-lifetime backstop -- provision right before running, and tear down at the end.

**keys.env:** the first executable statement below MUST stay `load_dotenv(...)` -- `smolbench.evals.ec2` captures its `EC2_*` config (here `EC2_EXPERIMENT_TAG=periodic-moe-induction`, `EC2_INSTANCE_TYPES=p5e.48xlarge`) from `os.environ` at import time, so keys.env must land before that module is ever imported.

In [ ]:
"""The following generates the Quiz all our models will be evaluated on."""

import string

import logging

from dotenv import load_dotenv
from pathlib import Path

logging.basicConfig(level=logging.INFO)
load_dotenv(Path.cwd() / "keys.env", verbose=True)

from smolbench.induction.periodic import (
    PeriodicConfig,
    Prompter,
    get_periodic_numeric_quiz,
    numeric_count_query_gen,
)

# Per-model names: keys of EC2_DEPLOY_SPECS in smolbench/evals/ec2.py (each
# is also vLLM's --served-model-name, sent in the OpenAI request body
# verbatim). This study holds the architecture class constant -- all three
# are Mixture-of-Experts -- so the tags name the MODEL, not a decode/cot/moe
# archetype. Each spec points at the highest-precision OFFICIAL release; all
# three repos are ungated.
MODEL_QWEN     = "qwen3.5-397b-a17b"           # Qwen/Qwen3.5-397B-A17B (BF16, MoE 397B/17B) -- ~794GB, p5e only
MODEL_NEMOTRON = "nemotron-3-super-120b-a12b"  # nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16 (BF16, MoE 120B/12B)
MODEL_GPTOSS   = "gpt-oss-120b"                # openai/gpt-oss-120b (MXFP4, MoE 120B/5.1B)

template = string.Template(
    "You are a precise integer counter.\n"
    "\n"
    "Task: answer the question below with a single integer and nothing else.\n"
    "\n"
    "Output format:\n"
    "Return exactly one integer and nothing else.\n"
    "Do not output any explanation, punctuation, quotes, or extra whitespace.\n"
    "Stop immediately after writing the integer.\n"
    "\n"
    "Context:\n"
    "There is a counting game. Positions are counted starting from 1. "
    "At each position, words are written according to the following rules:\n"
    "$positive_info\n"
    "Question:\n"
    "How many of the positions 1 through $seq_len include '$label'?"
)

# --- Replication setup -----------------------------------------------------
# Identical quizzes to notebooks/periodic (same template, n=9, BASE_SEED) so
# this all-MoE study is directly comparable to the archetype study. R=30 is
# carried over as a placeholder -- re-derive it with power_analysis.py once a
# pilot (seed 1776) exists.
BASE_SEED: int = 1776  # seed of the original preliminary run == replicate 0
INFO_TYPES: tuple[str, ...] = ("intens", "extens", "noise_intens")


def make_quizzes(seed: int) -> dict[str, tuple]:
    """Generates one replicate's three info-type quizzes, keyed by info type."""
    return dict(
        zip(
            INFO_TYPES,
            get_periodic_numeric_quiz(
                PeriodicConfig(
                    n=9,
                    labels=9,
                    seed=seed,
                ),
                Prompter(
                    template,
                    {},
                    numeric_count_query_gen,
                ),
            ),
        )
    )


# First-replicate aliases for the Prompt Validation cells below.
_base_quizzes: dict[str, tuple] = make_quizzes(BASE_SEED)
intens_quiz = _base_quizzes["intens"]
extens_quiz = _base_quizzes["extens"]
noise_intens_quiz = _base_quizzes["noise_intens"]


In [ ]:
# Builds this notebook's InductionExperiment: the replicate harness
# (results/{tag}_{info}/rep_{seed}.yaml, serialized right after grading) +
# the EC2 spot-instance lifecycle. See smolbench/evals/replicates.py and
# smolbench/induction/experiment.py.
from smolbench.induction.experiment import InductionExperiment

EXPERIMENT = InductionExperiment(
    notebook_dir="periodic_moe",
    archetype_tags={MODEL_QWEN: "qwen35", MODEL_NEMOTRON: "nemotron3", MODEL_GPTOSS: "gptoss"},
    make_quizzes=make_quizzes,
    n_replicates=30,
    base_seed=BASE_SEED,
    # Private EC2 state file so this study's instance record never clobbers
    # the periodic experiment's default .ec2_state.json (chromatic isolates
    # itself the same way). Pairs with EC2_EXPERIMENT_TAG in keys.env.
    state_file=".ec2_state_periodic_moe.json",
)


In [ ]:
# Provisions (or reattaches to) this experiment's EC2 spot instance --
# idempotent via the state file / smolbench:experiment tag, so it survives
# kernel restarts. Live AWS call; see the intro's cost note. keys.env pins
# p5e.48xlarge (BF16 Qwen3.5-397B needs the 1128 GB box).
state = EXPERIMENT.provision()


## Prompt Validation

In [ ]:
print(intens_quiz[0].prompt)

In [ ]:
print(extens_quiz[0].prompt)

In [ ]:
print(noise_intens_quiz[0].prompt)

## Qwen3.5-397B-A17B (MoE)
`Qwen/Qwen3.5-397B-A17B` -- BF16, 397B total / 17B active, Apache-2.0, ungated. ~794 GB at BF16 -> needs p5e (1128 GB; pinned in keys.env), does **not** fit p5. Hybrid Gated-DeltaNet + MoE, multimodal -> requires a recent vLLM image. The swap waits on the checkpoint download/load the first time; reruns hit the instance's warm HF cache.

In [ ]:
# Swaps the shared instance's vLLM to Qwen3.5 and runs every outstanding
# replicate across all info types; finished replicates are skipped on
# rerun. Single default mode (no reasoning contrast); 8192 completion tokens
# for headroom so a reasoning chain can close before the integer answer.
EXPERIMENT.run(MODEL_QWEN, extra_args={"max_completion_tokens": 8192})


In [ ]:
EXPERIMENT.summarize(MODEL_QWEN)

## Nemotron-3-Super-120B-A12B (MoE)
`nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16` -- BF16, 120B total / 12B active, ungated (verified via the HF models API). ~240 GB -> fits p5e easily. If it needs a reasoning system prompt (cf. `nemotron-ultra-253b`'s `"detailed thinking on"`), add `system_prompt` to its `EC2_DEPLOY_SPECS` entry so user prompts stay byte-identical across the trio.

In [ ]:
# Swaps vLLM to Nemotron-3-Super. Single default mode; 8192 completion
# tokens for headroom.
EXPERIMENT.run(MODEL_NEMOTRON, extra_args={"max_completion_tokens": 8192})


In [ ]:
EXPERIMENT.summarize(MODEL_NEMOTRON)

## GPT-OSS-120B (MoE)
`openai/gpt-oss-120b` -- native MXFP4 (its only/highest official precision), 120B total / 5.1B active, Apache-2.0, ungated. ~63 GB -> trivially fits p5e. Trained on the harmony response format (vLLM's chat template applies it); has low/medium/high reasoning-effort levels (default medium).

In [ ]:
# Swaps vLLM to GPT-OSS-120B. Single default mode (reasoning effort =
# medium); 8192 completion tokens for headroom.
EXPERIMENT.run(MODEL_GPTOSS, extra_args={"max_completion_tokens": 8192})


In [ ]:
EXPERIMENT.summarize(MODEL_GPTOSS)

# Teardown

In [ ]:
# Terminates the spot instance (and its EBS volume) and clears the private
# .ec2_state_periodic_moe.json. Also works after a kernel restart / lost
# state file: falls back to the smolbench:experiment=periodic-moe-induction tag.
EXPERIMENT.teardown()
